# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Horisyre/my-flyrank-ml-intern-starter-project/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [7]:
# ============================================================
# MULTICLASS IMPRESSION-TIER CLASSIFICATION
# ============================================================
#
# Target:
#     impression_tier
#
# Classes:
#     low, moderate, good, excellent
#
# Features:
#     clicks_30d
#     ctr_30d
#     health_score
#
# TRAINING:
#     2024-11-22 to 2026-04-30
#
# TESTING:
#     2026-05-01 to 2026-06-18
#
# Models:
#     1. Logistic Regression
#     2. Random Forest
#     3. Gradient Boosting
#
# Main ranking metric:
#     Precision@50 for "excellent"
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import os
import sys
import subprocess
import json


# ============================================================
# 2. LOAD DATA
# ============================================================

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:

    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            "requirements.txt"
        ],
        check=True
    )

else:

    while (
        not os.path.isdir("data/raw")
        and os.getcwd() != "/"
    ):
        os.chdir("..")


df = pd.read_csv(
    "data/raw/ranking_lifecycle.csv",
    low_memory=False
)


# ============================================================
# 3. CREATE CONTENT DATE
# ============================================================

df["content_date"] = df["content_created_at"].apply(
    lambda x: json.loads(x)["value"]
)

df["content_date"] = pd.to_datetime(
    df["content_date"],
    utc=True
)


# ============================================================
# 4. DEFINE TRAINING AND TESTING PERIODS
# ============================================================

# TRAIN:
# 22 November 2024 -> 30 April 2026

train_df = df[
    (df["content_date"] >= "2024-11-22") &
    (df["content_date"] < "2026-05-01")
].copy()


# TEST:
# 1 May 2026 -> 18 June 2026

test_df = df[
    (df["content_date"] >= "2026-05-01") &
    (df["content_date"] < "2026-06-19")
].copy()


# ============================================================
# 5. DISPLAY DATA PERIODS
# ============================================================

print("=" * 70)
print("DATA PERIODS")
print("=" * 70)

print("\nTraining period:")
print(
    train_df["content_date"].min(),
    "to",
    train_df["content_date"].max()
)

print("\nTesting period:")
print(
    test_df["content_date"].min(),
    "to",
    test_df["content_date"].max()
)

print("\nTraining rows:", len(train_df))
print("Testing rows:", len(test_df))


# ============================================================
# 6. SELECT THREE FEATURES
# ============================================================

features = [
    "clicks_30d",
    "ctr_30d",
    "health_score"
]

target = "impression_tier"


# ============================================================
# 7. CREATE X AND Y
# ============================================================

X_train = train_df[features].copy()
y_train = train_df[target].copy()

X_test = test_df[features].copy()
y_test = test_df[target].copy()


# Remove missing target values

train_valid = y_train.notna()
test_valid = y_test.notna()

X_train = X_train.loc[train_valid]
y_train = y_train.loc[train_valid]

X_test = X_test.loc[test_valid]
y_test = y_test.loc[test_valid]


# ============================================================
# 8. CHECK TARGET CLASSES BEFORE ENCODING
# ============================================================

print("\n" + "=" * 70)
print("TARGET CLASSES")
print("=" * 70)

print("\nTraining classes:")
print(sorted(y_train.unique()))

print("\nTesting classes:")
print(sorted(y_test.unique()))


# Check that all classes exist in training

expected_classes = [
    "low",
    "moderate",
    "good",
    "excellent"
]

missing_classes = set(expected_classes) - set(
    y_train.unique()
)

if missing_classes:

    raise ValueError(
        f"The following target classes are missing "
        f"from the training data: {missing_classes}"
    )


# ============================================================
# 9. ENCODE TARGET
# ============================================================

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(
    y_train
)

y_test_encoded = label_encoder.transform(
    y_test
)


print("\nEncoded classes:")
print(label_encoder.classes_)


# ============================================================
# 10. IDENTIFY EXCELLENT CLASS
# ============================================================

excellent_class = label_encoder.transform(
    ["excellent"]
)[0]

print(
    "\nExcellent encoded class:",
    excellent_class
)


# ============================================================
# 11. TARGET DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("TRAINING TARGET DISTRIBUTION")
print("=" * 70)

print(
    y_train.value_counts()
)


print("\n" + "=" * 70)
print("TESTING TARGET DISTRIBUTION")
print("=" * 70)

print(
    y_test.value_counts()
)


# ============================================================
# 12. DEFINE MODELS
# ============================================================

models = {

    "Logistic Regression": Pipeline([

        (
            "imputer",
            SimpleImputer(strategy="median")
        ),

        (
            "scaler",
            StandardScaler()
        ),

        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]),


    "Random Forest": Pipeline([

        (
            "imputer",
            SimpleImputer(strategy="median")
        ),

        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
                class_weight="balanced"
            )
        )
    ]),


    "Gradient Boosting": Pipeline([

        (
            "imputer",
            SimpleImputer(strategy="median")
        ),

        (
            "model",
            GradientBoostingClassifier(
                n_estimators=100,
                random_state=42
            )
        )
    ])
}


# ============================================================
# 13. PRECISION@50
# ============================================================

def precision_at_50(
    y_true,
    probabilities,
    target_class
):

    # Probability of being excellent
    target_probability = probabilities[
        :, target_class
    ]

    # Rank pages by probability
    ranked_indices = np.argsort(
        target_probability
    )[::-1]

    # Top 50
    k = min(50, len(ranked_indices))

    top_50_indices = ranked_indices[:k]

    # Actual classes
    top_50_actual = np.array(
        y_true
    )[top_50_indices]

    # Correct excellent predictions
    correct = np.sum(
        top_50_actual == target_class
    )

    precision = correct / k

    return (
        precision,
        correct,
        top_50_indices
    )


# ============================================================
# 14. TRAIN + EVALUATE
# ============================================================

results = {}


for name, model in models.items():

    print("\n")
    print("=" * 70)
    print(name.upper())
    print("=" * 70)


    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.fit(
        X_train,
        y_train_encoded
    )


    # --------------------------------------------------------
    # PREDICTIONS
    # --------------------------------------------------------

    y_pred = model.predict(
        X_test
    )

    y_proba = model.predict_proba(
        X_test
    )


    # --------------------------------------------------------
    # STANDARD METRICS
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_test_encoded,
        y_pred
    )

    weighted_precision = precision_score(
        y_test_encoded,
        y_pred,
        average="weighted",
        zero_division=0
    )

    weighted_recall = recall_score(
        y_test_encoded,
        y_pred,
        average="weighted",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_test_encoded,
        y_pred,
        average="weighted",
        zero_division=0
    )


    # --------------------------------------------------------
    # PRECISION@50
    # --------------------------------------------------------

    p50, correct_50, top_50_indices = precision_at_50(

        y_test_encoded,

        y_proba,

        excellent_class
    )


    # --------------------------------------------------------
    # SAVE RESULTS
    # --------------------------------------------------------

    results[name] = {

        "model": model,

        "accuracy": accuracy,

        "weighted_precision":
            weighted_precision,

        "weighted_recall":
            weighted_recall,

        "weighted_f1":
            weighted_f1,

        "precision_at_50":
            p50,

        "correct_at_50":
            correct_50,

        "top_50_indices":
            top_50_indices,

        "predictions":
            y_pred,

        "probabilities":
            y_proba
    }


    # --------------------------------------------------------
    # PRINT RESULTS
    # --------------------------------------------------------

    print(
        f"\nAccuracy:           {accuracy:.4f}"
    )

    print(
        f"Weighted Precision: {weighted_precision:.4f}"
    )

    print(
        f"Weighted Recall:    {weighted_recall:.4f}"
    )

    print(
        f"Weighted F1:        {weighted_f1:.4f}"
    )

    print(
        f"Precision@50:       {p50:.4f}"
    )

    print(
        f"Correct Top 50:     {correct_50}/50"
    )


    # --------------------------------------------------------
    # CLASSIFICATION REPORT
    # --------------------------------------------------------

    print("\nClassification Report:")

    print(
        classification_report(
            y_test_encoded,
            y_pred,
            target_names=label_encoder.classes_,
            zero_division=0
        )
    )


# ============================================================
# 15. MODEL COMPARISON
# ============================================================

performance = pd.DataFrame({

    "Model": list(results.keys()),

    "Accuracy": [
        results[m]["accuracy"]
        for m in results
    ],

    "Weighted Precision": [
        results[m]["weighted_precision"]
        for m in results
    ],

    "Weighted Recall": [
        results[m]["weighted_recall"]
        for m in results
    ],

    "Weighted F1": [
        results[m]["weighted_f1"]
        for m in results
    ],

    "Precision@50": [
        results[m]["precision_at_50"]
        for m in results
    ],

    "Correct Top 50": [
        results[m]["correct_at_50"]
        for m in results
    ]
})


performance = performance.sort_values(
    "Precision@50",
    ascending=False
).reset_index(drop=True)


print("\n")
print("=" * 70)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 70)

display(
    performance.style.format({

        "Accuracy": "{:.3f}",

        "Weighted Precision": "{:.3f}",

        "Weighted Recall": "{:.3f}",

        "Weighted F1": "{:.3f}",

        "Precision@50": "{:.2%}",

        "Correct Top 50": "{:.0f}"
    })
)


# ============================================================
# 16. SHOW TOP 50 FOR EACH MODEL
# ============================================================

for name, result in results.items():

    print("\n")
    print("=" * 70)
    print(f"TOP 50 — {name}")
    print("=" * 70)


    top_indices = result[
        "top_50_indices"
    ]


    # Get corresponding rows
    top_50 = test_df.loc[
        X_test.iloc[top_indices].index
    ].copy()


    # Probability of excellent
    top_50[
        "excellent_probability"
    ] = result[
        "probabilities"
    ][
        top_indices,
        excellent_class
    ]


    # Predicted class
    predicted_classes = result[
        "predictions"
    ][top_indices]


    top_50[
        "predicted_tier"
    ] = label_encoder.inverse_transform(
        predicted_classes
    )


    # Correct / incorrect
    top_50[
        "correct"
    ] = (
        top_50["impression_tier"]
        == "excellent"
    )


    # Sort highest confidence first
    top_50 = top_50.sort_values(
        "excellent_probability",
        ascending=False
    )


    display(
        top_50[
            [
                "content_date",
                "clicks_30d",
                "ctr_30d",
                "health_score",
                "impression_tier",
                "predicted_tier",
                "excellent_probability",
                "correct"
            ]
        ].head(50)
    )

DATA PERIODS

Training period:
2024-11-22 14:48:37+00:00 to 2026-04-30 23:47:45.921040+00:00

Testing period:
2026-05-01 00:03:42.067337+00:00 to 2026-06-18 13:39:00.956924+00:00

Training rows: 183844
Testing rows: 21903

TARGET CLASSES

Training classes:
['excellent', 'good', 'low', 'moderate']

Testing classes:
['excellent', 'good', 'low', 'moderate']

Encoded classes:
['excellent' 'good' 'low' 'moderate']

Excellent encoded class: 0

TRAINING TARGET DISTRIBUTION
impression_tier
low          92254
moderate     59550
good         28239
excellent     3801
Name: count, dtype: int64

TESTING TARGET DISTRIBUTION
impression_tier
low          11295
moderate      7558
good          2857
excellent      193
Name: count, dtype: int64


LOGISTIC REGRESSION

Accuracy:           0.8768
Weighted Precision: 0.8819
Weighted Recall:    0.8768
Weighted F1:        0.8735
Precision@50:       1.0000
Correct Top 50:     50/50

Classification Report:
              precision    recall  f1-score   support

 

,Model,Accuracy,Weighted Precision,Weighted Recall,Weighted F1,Precision@50,Correct Top 50
0,Logistic Regression,0.877,0.882,0.877,0.873,100.00%,50
1,Random Forest,0.831,0.850,0.831,0.832,100.00%,50
2,Gradient Boosting,0.889,0.892,0.889,0.887,100.00%,50




TOP 50 — Logistic Regression


,content_date,clicks_30d,ctr_30d,health_score,impression_tier,predicted_tier,excellent_probability,correct
168680,2026-05-03 00:06:45.874154+00:00,366,1.25,70,excellent,excellent,1.000000,True
173246,2026-05-21 00:06:45.397443+00:00,441,2.14,70,excellent,excellent,1.000000,True
172892,2026-05-15 00:05:42.610730+00:00,438,2.18,70,excellent,excellent,1.000000,True
200997,2026-06-07 00:11:36.929668+00:00,88396,52.73,85,excellent,excellent,1.000000,True
172040,2026-05-01 00:31:05.178920+00:00,387,1.30,60,excellent,excellent,1.000000,True
168700,2026-05-03 00:11:13.047600+00:00,626,1.07,65,excellent,excellent,1.000000,True
200994,2026-06-06 00:14:26.875614+00:00,97954,53.22,85,excellent,excellent,1.000000,True
168650,2026-05-02 00:09:37.418579+00:00,513,2.14,75,excellent,excellent,1.000000,True
174180,2026-06-05 00:18:58.679566+00:00,900,6.86,75,excellent,excellent,1.000000,True
172664,2026-05-11 00:09:28.920670+00:00,827,2.09,60,excellent,excellent,1.000000,True




TOP 50 — Random Forest


,content_date,clicks_30d,ctr_30d,health_score,impression_tier,predicted_tier,excellent_probability,correct
169366,2026-05-26 00:13:04.497509+00:00,55,0.46,65,excellent,excellent,1.0,True
168672,2026-05-03 00:05:14.049507+00:00,94,0.54,65,excellent,excellent,1.0,True
172051,2026-05-02 00:12:34.612733+00:00,53,0.12,60,excellent,excellent,1.0,True
168971,2026-05-14 00:13:33.166729+00:00,139,0.33,65,excellent,excellent,1.0,True
172319,2026-05-04 00:25:42.004993+00:00,148,0.16,60,excellent,excellent,1.0,True
168896,2026-05-12 00:16:18.458817+00:00,51,0.32,65,excellent,excellent,1.0,True
168893,2026-05-12 00:15:15.736126+00:00,19,0.15,65,excellent,excellent,1.0,True
168892,2026-05-12 00:15:03.936325+00:00,89,0.72,65,excellent,excellent,1.0,True
172359,2026-05-07 00:06:21.336292+00:00,68,0.14,60,excellent,excellent,1.0,True
168890,2026-05-12 00:14:47.592362+00:00,243,0.53,65,excellent,excellent,1.0,True




TOP 50 — Gradient Boosting


,content_date,clicks_30d,ctr_30d,health_score,impression_tier,predicted_tier,excellent_probability,correct
169531,2026-05-31 00:09:27.898555+00:00,25,0.04,70,excellent,excellent,0.999998,True
169000,2026-05-15 00:11:43.480459+00:00,20,0.07,65,excellent,excellent,0.999994,True
172954,2026-05-16 00:07:26.660377+00:00,45,0.06,50,excellent,excellent,0.999974,True
168745,2026-05-08 00:17:00.158845+00:00,7,0.02,65,excellent,excellent,0.999959,True
173023,2026-05-17 00:07:11.390591+00:00,214,0.10,60,excellent,excellent,0.999893,True
171975,2026-05-01 00:14:53.257047+00:00,158,0.14,60,excellent,excellent,0.999830,True
169207,2026-05-21 00:14:35.991084+00:00,13,0.06,65,excellent,excellent,0.999806,True
172319,2026-05-04 00:25:42.004993+00:00,148,0.16,60,excellent,excellent,0.999802,True
179500,2026-05-17 00:07:24.243086+00:00,26,0.10,80,excellent,excellent,0.999780,True
172650,2026-05-11 00:08:46.378048+00:00,145,0.25,60,excellent,excellent,0.999657,True


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.